In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ==============================================================
# 1) PARAMETER & SETUP
# ==============================================================
lam            = 1064e-9      # Wellenlänge [m]
f_lens         = 5.0          # Brennweite der Linse [m]
TWO_W_START    = 11.0e-3      # Ziel-Durchmesser D4sigma bei z=0 [m]
N              = 1024         # Auflösung
L              = 60e-3        # Simulationsfenster [m]
dx             = L / N

# Propagations-Bereich (Fokus liegt bei 5m, wir plotten bis 8m)
z_max          = 8.0          
z_steps        = 100
z_array        = np.linspace(0, z_max, z_steps)

# Koordinatengitter
x = (np.arange(N) - N/2) * dx
y = x
X, Y = np.meshgrid(x, y)
R = np.sqrt(X**2 + Y**2)

# ==============================================================
# 2) FUNKTIONEN
# ==============================================================

def get_d4sigma_mm(I, spatial_dx):
    """Berechnet den D4sigma-Durchmesser in mm."""
    total = np.sum(I)
    if total <= 0: return 0.0
    P = I / total
    cx = np.sum(X * P)
    cy = np.sum(Y * P)
    
    # Varianz (zweites Moment)
    sigma_sq = 0.5 * (np.sum(P * (X - cx)**2) + np.sum(P * (Y - cy)**2))
    return 4 * np.sqrt(max(sigma_sq, 0)) * 1000

def propagate_asm(U, dz):
    """Angular Spectrum Propagation (ASM)."""
    k = 2 * np.pi / lam
    fx = np.fft.fftfreq(N, d=dx)
    fy = np.fft.fftfreq(N, d=dx)
    FX, FY = np.meshgrid(fx, fy)
    
    kz = np.sqrt((k+0j)**2 - (2*np.pi*FX)**2 - (2*np.pi*FY)**2)
    H = np.exp(1j * kz * dz)
    
    return np.fft.ifft2(np.fft.fft2(U) * H)

# ==============================================================
# 3) SIMULATION (Super-Gauß n=100 mit Linse)
# ==============================================================

# A) Erzeugung Startfeld (n=100)
# Kalibrierung: w0 so wählen, dass D4sigma bei z=0 exakt TWO_W_START ist
w0 = (TWO_W_START / 2) / 1.154 
U0 = np.exp(-(R/w0)**100).astype(complex)

# B) LINSE ANWENDEN (Phase aufprägen)
k = 2 * np.pi / lam
phase_lens = np.exp(-1j * k * R**2 / (2 * f_lens))
U_after_lens = U0 * phase_lens

# C) Propagation durchführen
d_vals = []
print(f"Simuliere Fokusverlauf für Super-Gauß n=100...")

for zi in z_array:
    Uz = propagate_asm(U_after_lens, zi)
    d_vals.append(get_d4sigma_mm(np.abs(Uz)**2, dx))

d_vals = np.array(d_vals)

# Relative Änderung [%] bezogen auf den Startwert d_vals[0] (bei z=0)
d_start = d_vals[0]
slope_pct = (d_vals - d_start) / d_start * 100

# ==============================================================
# 4) PLOTTING (Mit angeforderter Beschriftung)
# ==============================================================
fig, ax1 = plt.subplots(figsize=(11, 6))

# Linke Achse: Absolute Werte
color_abs = 'navy'
ax1.plot(z_array, d_vals, color=color_abs, lw=2.5, label='Strahl-Kaustik (mm)')
ax1.axvline(f_lens, color='black', linestyle=':', label=f'Fokus-Ebene (z={f_lens}m)')

ax1.set_xlabel("Propagationsdistanz z [m]")
ax1.set_ylabel("Strahldurchmesser 2w [mm] (D4σ)", color=color_abs, fontsize=12)
ax1.tick_params(axis='y', labelcolor=color_abs)
ax1.grid(True, linestyle=':', alpha=0.6)

# Rechte Achse: Relative Steigung / Änderung
ax2 = ax1.twinx()
color_slope = 'tab:red'
ax2.plot(z_array, slope_pct, color=color_slope, ls='--', lw=2, label='Relative Änderung (%)')
ax2.set_ylabel("Relative Änderung [%] (bezogen auf Start-2w)", color=color_slope, fontsize=12)
ax2.tick_params(axis='y', labelcolor=color_slope)
ax2.axhline(0, color='gray', lw=1, alpha=0.5)

# Kombinierte Legende
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right')

plt.title(f"Theoretische Propagation (Linse f={f_lens}m): Super-Gauß n=100", pad=15)
fig.tight_layout()
plt.show()

# Fokus-Analyse in der Konsole
idx_fokus = np.argmin(d_vals)
print(f"Minimaler Durchmesser (Fokus) bei z={z_array[idx_fokus]:.2f}m: {d_vals[idx_fokus]:.4f} mm")

Simuliere Fokusverlauf für Super-Gauß n=100...
